In [ ]:
%load_ext sql

# Connect SQLMagic directly to the project DuckDB file.
%sql duckdb:///f:/Project/games/jt3/data/jt3.duckdb

%config SqlMagic.autopolars = True

# %config SqlMagic.named_parameters = "enabled"

In [ ]:
%%sql rows <<
SELECT DISTINCT c.text AS clue_text
FROM clues AS c
ORDER BY c.game_id DESC, c.round_index, c.category_index, c.clue_order

In [ ]:
from sentence_transformers import SentenceTransformer

MODELS = {
    "all_MiniLM_L6_v2": dict(model_name_or_path="sentence-transformers/all-MiniLM-L6-v2", device='cuda', backend="onnx", model_kwargs={"provider": "CUDAExecutionProvider"}),
    "qwen3_embedding_06B": dict(model_name_or_path="Qwen/Qwen3-Embedding-0.6B", device='cuda'),
    "qwen3_embedding_06B_trunc_32": dict(model_name_or_path="Qwen/Qwen3-Embedding-0.6B", device='cuda', truncate_dim=32),
}

model = SentenceTransformer(**MODELS["all_MiniLM_L6_v2"])

In [ ]:
clues = rows['clue_text'].to_list()
embeddings = model.encode(clues, batch_size=128, show_progress_bar=True)

In [ ]:
import polars as pl
import duckdb

dim = embeddings.shape[1]
df = pl.DataFrame({
    "clue_text": clues,
    "embedding": embeddings.tolist(),
}).unique(subset=["clue_text"], keep="first")

# Close SQLMagic's connection so we can get a write lock with native duckdb.
%sql --close duckdb:///f:/Project/games/jt3/data/jt3.duckdb

con = duckdb.connect("f:/Project/games/jt3/data/jt3.duckdb")
con.sql("DROP TABLE IF EXISTS clue_embeddings")
con.sql(f"CREATE TABLE clue_embeddings (clue_text TEXT PRIMARY KEY, embedding FLOAT[{dim}] NOT NULL)")
con.sql("INSERT INTO clue_embeddings SELECT clue_text, embedding FROM df")
con.close()

print(f"Saved {len(df)} clue embeddings ({len(clues) - len(df)} duplicates removed)")

In [ ]:
# Reopen for subsequent %sql cells.
%sql duckdb:///f:/Project/games/jt3/data/jt3.duckdb

%sql SELECT * FROM clue_embeddings LIMIT 5

In [ ]:
%%sql response_rows <<
SELECT DISTINCT c.correct_response AS response_text
FROM clues AS c
WHERE c.correct_response IS NOT NULL
ORDER BY c.game_id DESC, c.round_index, c.category_index, c.clue_order

In [ ]:
responses = response_rows['response_text'].to_list()
response_embeddings = model.encode(responses, batch_size=128, show_progress_bar=True)

In [ ]:
rdim = response_embeddings.shape[1]
rdf = pl.DataFrame({
    "response_text": responses,
    "embedding": response_embeddings.tolist(),
}).unique(subset=["response_text"], keep="first")

%sql --close duckdb:///f:/Project/games/jt3/data/jt3.duckdb

con = duckdb.connect("f:/Project/games/jt3/data/jt3.duckdb")
con.sql("DROP TABLE IF EXISTS response_embeddings")
con.sql(f"CREATE TABLE response_embeddings (response_text TEXT PRIMARY KEY, embedding FLOAT[{rdim}] NOT NULL)")
con.sql("INSERT INTO response_embeddings SELECT response_text, embedding FROM rdf")
con.close()

print(f"Saved {len(rdf)} response embeddings ({len(responses) - len(rdf)} duplicates removed)")

In [ ]:
%sql duckdb:///f:/Project/games/jt3/data/jt3.duckdb

%sql SELECT * FROM response_embeddings LIMIT 5

In [ ]:
%%sql ctx_rows <<
SELECT
    c.correct_response,
    cat.name AS category,
    c.text AS clue
FROM clues c
JOIN categories cat ON c.game_id = cat.game_id
    AND c.round_index = cat.round_index
    AND c.category_index = cat.category_index
WHERE c.correct_response IS NOT NULL

In [8]:
import numpy as np

# Build "Category: Clue → Response" strings per response
response_contexts = {}
for row in ctx_rows.iter_rows(named=True):
    resp = row["correct_response"]
    ctx = f"{row['category']}: {row['clue']} → {resp}"
    response_contexts.setdefault(resp, []).append(ctx)

# Flatten for batch encoding
all_strings = []
string_to_response = []
ctx_response_keys = list(response_contexts.keys())

for i, resp in enumerate(ctx_response_keys):
    for ctx in response_contexts[resp]:
        all_strings.append(ctx)
        string_to_response.append(i)

print(f"Encoding {len(all_strings)} context strings for {len(ctx_response_keys)} responses...")
ctx_embs = model.encode(all_strings, batch_size=128, show_progress_bar=True)

# Average embeddings per response, then L2-normalize
ctx_avg = np.zeros((len(ctx_response_keys), ctx_embs.shape[1]))
counts = np.zeros(len(ctx_response_keys))
for idx, emb in zip(string_to_response, ctx_embs):
    ctx_avg[idx] += emb
    counts[idx] += 1
ctx_avg /= counts[:, np.newaxis]
ctx_avg /= np.linalg.norm(ctx_avg, axis=1, keepdims=True)

Encoding 296438 context strings for 106909 responses...


Batches:   0%|          | 0/2316 [00:00<?, ?it/s]

In [15]:
import polars as pl
import duckdb
import json

ctx_dim = ctx_avg.shape[1]
cdf = pl.DataFrame({
    "response_text": ctx_response_keys,
    "context_texts": [json.dumps(response_contexts[r]) for r in ctx_response_keys],
    "embedding": ctx_avg.tolist(),
})

%sql --close duckdb:///f:/Project/games/jt3/data/jt3.duckdb

con = duckdb.connect("f:/Project/games/jt3/data/jt3.duckdb")
con.sql("DROP TABLE IF EXISTS contextual_response_embeddings")
con.sql(f"""CREATE TABLE contextual_response_embeddings (
    response_text TEXT PRIMARY KEY,
    context_texts JSON NOT NULL,
    embedding FLOAT[{ctx_dim}] NOT NULL
)""")
con.sql("INSERT INTO contextual_response_embeddings SELECT response_text, context_texts, embedding FROM cdf")
con.close()

print(f"Saved {len(cdf)} contextual response embeddings")

Saved 106909 contextual response embeddings


In [16]:
%sql duckdb:///f:/Project/games/jt3/data/jt3.duckdb

%sql SELECT * FROM contextual_response_embeddings LIMIT 5

Connecting and switching to connection 'duckdb:///f:/Project/games/jt3/data/jt3.duckdb'

Running query in 'duckdb:///f:/Project/games/jt3/data/jt3.duckdb'

response_text,context_texts,embedding
str,str,"array[f32, 384]"
"""a man""","""[""THE CIVIL WAR: Loreta Velasq…","[-0.062542, 0.026462, … -0.077282]"
"""the North""","""[""THE CIVIL WAR: The Living Hi…","[-0.030526, 0.076277, … 0.051457]"
"""north""","""[""THE CIVIL WAR: On Richmond's…","[0.01151, 0.046514, … -0.02081]"
"""""Taps""""","""[""THE CIVIL WAR: While occupyi…","[-0.031203, -0.000702, … 0.041915]"
"""the Emancipation Proclamation""","""[""THE CIVIL WAR: Surprisingly,…","[-0.084008, 0.084761, … -0.031254]"


In [ ]:
PROMPT = "Represent this trivia answer for finding topically related answers: "

prompted_embs = model.encode(responses, prompt=PROMPT, batch_size=128, show_progress_bar=True)
prompted_embs /= np.linalg.norm(prompted_embs, axis=1, keepdims=True)

print(f"Encoded {len(responses)} prompted response embeddings")

In [ ]:
pdim = prompted_embs.shape[1]
pdf = pl.DataFrame({
    "response_text": responses,
    "embedding": prompted_embs.tolist(),
}).unique(subset=["response_text"], keep="first")

%sql --close duckdb:///f:/Project/games/jt3/data/jt3.duckdb

con = duckdb.connect("f:/Project/games/jt3/data/jt3.duckdb")
con.sql("DROP TABLE IF EXISTS prompted_response_embeddings")
con.sql(f"CREATE TABLE prompted_response_embeddings (response_text TEXT PRIMARY KEY, embedding FLOAT[{pdim}] NOT NULL)")
con.sql("INSERT INTO prompted_response_embeddings SELECT response_text, embedding FROM pdf")
con.close()

print(f"Saved {len(pdf)} prompted response embeddings")

In [ ]:
%sql duckdb:///f:/Project/games/jt3/data/jt3.duckdb

%sql SELECT * FROM prompted_response_embeddings LIMIT 5